# NHL Fantasy Optimizer v2

**Single-notebook optimizer for weekly salary-cap fantasy hockey.**

### How to Use
1. Make sure `nhl_players_2025_26.csv` is in the same directory (from PuckPedia scrape in `Salary Cap.ipynb`)
2. Run all cells top-to-bottom
3. The optimizer will auto-detect the current week's schedule and output the optimal 12-player lineup

### Data Sources
- **Player stats & schedule**: NHL API (free, no auth)
- **Salary data**: CSV from PuckPedia (refresh via `Salary Cap.ipynb` if needed)
- **Injuries**: CBS Sports scrape

In [1]:
# Cell 2 — Imports & Auto-Config

import pandas as pd
import numpy as np
import requests
import sqlite3
import pulp
import unicodedata
import time
from datetime import date, datetime, timedelta
from io import StringIO
from pathlib import Path
from bs4 import BeautifulSoup

# Auto-detect season
today = date.today()
season_start_year = today.year if today.month >= 9 else today.year - 1
SEASON_INT = f"{season_start_year}{season_start_year + 1}"

# Find salary CSV
SALARY_CSV = next(Path('.').glob('nhl_players_*.csv'), None)
if SALARY_CSV is None:
    raise FileNotFoundError("No salary CSV found. Run Salary Cap.ipynb first.")
print(f"Season: {SEASON_INT}")
print(f"Salary CSV: {SALARY_CSV}")

# Optimizer constraints
MAX_COST = 70.5
MIN_COST = MAX_COST * 0.9
NUM_FORWARDS = 6
NUM_DEFENSEMEN = 4
NUM_GOALIES = 2
MAX_PLAYERS_PER_TEAM = 5

# Cache
CACHE_HOURS = 12
DB_PATH = 'nhl_optimizer.db'

Season: 20252026
Salary CSV: nhl_players_2025_26.csv


In [2]:
# Cell 3 — Team Abbreviation Mappings

# Maps full team names (as used in salary CSV) → NHL API 3-letter codes
FULL_NAME_TO_ABBREV = {
    'Anaheim Ducks': 'ANA',
    'Boston Bruins': 'BOS',
    'Buffalo Sabres': 'BUF',
    'Calgary Flames': 'CGY',
    'Carolina Hurricanes': 'CAR',
    'Chicago Blackhawks': 'CHI',
    'Colorado Avalanche': 'COL',
    'Columbus Blue Jackets': 'CBJ',
    'Dallas Stars': 'DAL',
    'Detroit Red Wings': 'DET',
    'Edmonton Oilers': 'EDM',
    'Florida Panthers': 'FLA',
    'Los Angeles Kings': 'LAK',
    'Minnesota Wild': 'MIN',
    'Montréal Canadiens': 'MTL',
    'Montreal Canadiens': 'MTL',
    'Nashville Predators': 'NSH',
    'New Jersey Devils': 'NJD',
    'New York Islanders': 'NYI',
    'New York Rangers': 'NYR',
    'Ottawa Senators': 'OTT',
    'Philadelphia Flyers': 'PHI',
    'Pittsburgh Penguins': 'PIT',
    'San Jose Sharks': 'SJS',
    'Seattle Kraken': 'SEA',
    'St. Louis Blues': 'STL',
    'St Louis Blues': 'STL',
    'Tampa Bay Lightning': 'TBL',
    'Toronto Maple Leafs': 'TOR',
    'Utah Hockey Club': 'UTA',
    'Utah Mammoth': 'UTA',
    'Vancouver Canucks': 'VAN',
    'Vegas Golden Knights': 'VGK',
    'Washington Capitals': 'WSH',
    'Winnipeg Jets': 'WPG',
}

# CBS Sports uses shortened team names — map those too
CBS_TEAM_TO_ABBREV = {
    'Anaheim': 'ANA',
    'Arizona': 'UTA',
    'Boston': 'BOS',
    'Buffalo': 'BUF',
    'Calgary': 'CGY',
    'Carolina': 'CAR',
    'Chicago': 'CHI',
    'Colorado': 'COL',
    'Columbus': 'CBJ',
    'Dallas': 'DAL',
    'Detroit': 'DET',
    'Edmonton': 'EDM',
    'Florida': 'FLA',
    'Los Angeles': 'LAK',
    'Minnesota': 'MIN',
    'Montreal': 'MTL',
    'Nashville': 'NSH',
    'New Jersey': 'NJD',
    'N.Y. Islanders': 'NYI',
    'N.Y. Rangers': 'NYR',
    'Ottawa': 'OTT',
    'Philadelphia': 'PHI',
    'Pittsburgh': 'PIT',
    'San Jose': 'SJS',
    'Seattle': 'SEA',
    'St. Louis': 'STL',
    'Tampa Bay': 'TBL',
    'Toronto': 'TOR',
    'Utah': 'UTA',
    'Vancouver': 'VAN',
    'Vegas': 'VGK',
    'Washington': 'WSH',
    'Winnipeg': 'WPG',
}

# All 32 NHL API team codes
ALL_TEAMS = sorted(set(FULL_NAME_TO_ABBREV.values()))

In [3]:
# Cell 4 — SQLite Cache Layer

def get_db():
    """Get SQLite connection and create tables if needed."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS cache_metadata (
        table_name TEXT PRIMARY KEY,
        updated_at TEXT
    )''')
    conn.execute('''CREATE TABLE IF NOT EXISTS player_stats (
        player_name TEXT,
        team TEXT,
        position TEXT,
        games_played INTEGER,
        goals INTEGER,
        assists INTEGER,
        shots INTEGER,
        avg_toi_seconds REAL,
        goals_per_game REAL,
        assists_per_game REAL
    )''')
    conn.execute('''CREATE TABLE IF NOT EXISTS standings (
        team TEXT PRIMARY KEY,
        team_name TEXT,
        point_pctg REAL
    )''')
    conn.commit()
    return conn


def is_cache_fresh(table_name):
    """Check if cached data is less than CACHE_HOURS old."""
    conn = get_db()
    row = conn.execute(
        'SELECT updated_at FROM cache_metadata WHERE table_name = ?',
        (table_name,)
    ).fetchone()
    conn.close()
    if row is None:
        return False
    updated = datetime.fromisoformat(row[0])
    return (datetime.now() - updated).total_seconds() < CACHE_HOURS * 3600


def set_cache_timestamp(conn, table_name):
    """Update cache timestamp for a table."""
    conn.execute(
        'INSERT OR REPLACE INTO cache_metadata (table_name, updated_at) VALUES (?, ?)',
        (table_name, datetime.now().isoformat())
    )
    conn.commit()

In [4]:
# Cell 5 — NHL API Fetch Functions

NHL_API_BASE = 'https://api-web.nhle.com/v1'


def normalize_name(name):
    """Normalize unicode characters in player names."""
    return unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')


def fetch_all_player_stats(min_gp=10):
    """Fetch skater stats from all 32 teams via NHL API. Returns DataFrame."""
    if is_cache_fresh('player_stats'):
        conn = get_db()
        df = pd.read_sql('SELECT * FROM player_stats', conn)
        conn.close()
        print(f"Using cached player stats ({len(df)} players)")
        return df

    print(f"Fetching player stats from NHL API for season {SEASON_INT}...")
    all_players = []

    for team in ALL_TEAMS:
        url = f"{NHL_API_BASE}/club-stats/{team}/{SEASON_INT}/2"
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  Warning: Failed to fetch {team}: {e}")
            continue

        for skater in data.get('skaters', []):
            gp = skater.get('gamesPlayed', 0)
            if gp < min_gp:
                continue

            first = skater.get('firstName', {}).get('default', '')
            last = skater.get('lastName', {}).get('default', '')
            name = normalize_name(f"{first} {last}").strip()

            pos_code = skater.get('positionCode', 'C')
            position = 'D' if pos_code == 'D' else 'F'

            goals = skater.get('goals', 0)
            assists = skater.get('assists', 0)
            shots = skater.get('shots', 0)
            toi = skater.get('avgTimeOnIcePerGame', 0)  # seconds

            all_players.append({
                'player_name': name,
                'team': team,
                'position': position,
                'games_played': gp,
                'goals': goals,
                'assists': assists,
                'shots': shots,
                'avg_toi_seconds': toi,
                'goals_per_game': goals / gp,
                'assists_per_game': assists / gp,
            })

        time.sleep(0.1)  # rate limiting

    df = pd.DataFrame(all_players)
    print(f"Fetched {len(df)} players from {len(ALL_TEAMS)} teams")

    # Cache to SQLite
    conn = get_db()
    conn.execute('DELETE FROM player_stats')
    df.to_sql('player_stats', conn, if_exists='append', index=False)
    set_cache_timestamp(conn, 'player_stats')
    conn.close()

    return df


def fetch_standings():
    """Fetch current standings from NHL API. Returns DataFrame with team abbrev and point_pctg."""
    if is_cache_fresh('standings'):
        conn = get_db()
        df = pd.read_sql('SELECT * FROM standings', conn)
        conn.close()
        print(f"Using cached standings ({len(df)} teams)")
        return df

    print("Fetching standings from NHL API...")
    resp = requests.get(f"{NHL_API_BASE}/standings/now", timeout=10)
    resp.raise_for_status()
    data = resp.json()

    teams = []
    for entry in data.get('standings', []):
        abbrev = entry.get('teamAbbrev', {}).get('default', '')
        name = entry.get('teamName', {}).get('default', '')
        pctg = entry.get('pointPctg', 0.5)
        teams.append({'team': abbrev, 'team_name': name, 'point_pctg': pctg})

    df = pd.DataFrame(teams)
    print(f"Fetched standings for {len(df)} teams")

    # Cache
    conn = get_db()
    conn.execute('DELETE FROM standings')
    df.to_sql('standings', conn, if_exists='append', index=False)
    set_cache_timestamp(conn, 'standings')
    conn.close()

    return df


def fetch_weekly_schedule(start_date=None):
    """Fetch this week's schedule. Returns (games_count, opponents) dicts keyed by team abbrev."""
    if start_date is None:
        start_date = date.today()
    elif isinstance(start_date, str):
        start_date = date.fromisoformat(start_date)

    print(f"Fetching schedule starting {start_date}...")
    resp = requests.get(f"{NHL_API_BASE}/schedule/{start_date.isoformat()}", timeout=10)
    resp.raise_for_status()
    data = resp.json()

    games_count = {}   # team → number of games
    opponents = {}     # team → list of opponent abbrevs

    for day in data.get('gameWeek', []):
        game_date = date.fromisoformat(day['date'])
        if game_date < start_date:
            continue
        for game in day.get('games', []):
            away = game.get('awayTeam', {}).get('abbrev', '')
            home = game.get('homeTeam', {}).get('abbrev', '')
            if not away or not home:
                continue

            games_count[away] = games_count.get(away, 0) + 1
            games_count[home] = games_count.get(home, 0) + 1
            opponents.setdefault(away, []).append(home)
            opponents.setdefault(home, []).append(away)

    teams_playing = len(games_count)
    total_games = sum(games_count.values()) // 2
    print(f"Found {total_games} games this week, {teams_playing} teams playing")
    return games_count, opponents

In [5]:
# Cell 6 — Schedule Multipliers

def calculate_multipliers(standings_df, opponents):
    """Calculate schedule strength multiplier per team.
    
    For each team, average (0.5 / opponent_pointPctg) across their opponents.
    Weaker opponents → higher multiplier → more projected points.
    """
    pctg_lookup = dict(zip(standings_df['team'], standings_df['point_pctg']))
    multipliers = {}

    for team, opps in opponents.items():
        opp_mults = []
        for opp in opps:
            pctg = pctg_lookup.get(opp, 0.5)
            opp_mults.append(0.5 / pctg if pctg > 0 else 1.8)
        multipliers[team] = np.mean(opp_mults) if opp_mults else 1.0

    return multipliers

In [6]:
# Cell 7 — Injury Scraper

import re

def clean_player_name(name):
    """Clean CBS Sports player name format to standard 'First Last'.
    
    CBS format: 'J. CarlsonJohn Carlson' — abbreviated name concatenated with full name.
    We extract the full name by finding where the uppercase letter starts after the abbreviated part.
    """
    # Pattern: abbreviated name (e.g. "J. Carlson") immediately followed by full name ("John Carlson")
    # The full name starts with an uppercase letter right after the last word of the abbreviated name
    match = re.search(r'[a-z]([A-Z])', name)
    if match:
        full_name = name[match.start() + 1:]
        return full_name.strip()
    return name.strip()


def get_current_injuries():
    """Scrape current NHL injuries from CBS Sports. Returns DataFrame."""
    try:
        url = "https://www.cbssports.com/nhl/injuries/"
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        injury_dataframes = []

        for wrapper in soup.find_all('div', class_='TableBaseWrapper'):
            team_name_tag = wrapper.find('span', class_='TeamName')
            if team_name_tag:
                team_name = team_name_tag.get_text(strip=True)
                table = wrapper.find('table', class_='TableBase-table')
                if table:
                    df = pd.read_html(StringIO(str(table)))[0]
                    # Map CBS short team name to abbreviation
                    df['Team'] = CBS_TEAM_TO_ABBREV.get(team_name, team_name)
                    injury_dataframes.append(df)

        if not injury_dataframes:
            print("No injury data found on CBS Sports")
            return pd.DataFrame(columns=['Player', 'Team'])

        injuries_df = pd.concat(injury_dataframes, ignore_index=True)
        injuries_df['Player'] = injuries_df['Player'].apply(clean_player_name)
        print(f"Found {len(injuries_df)} injured players")
        return injuries_df

    except Exception as e:
        print(f"Warning: Could not fetch injuries ({e}). Continuing without injury data.")
        return pd.DataFrame(columns=['Player', 'Team'])

In [7]:
# Cell 8 — Salary Data Loader

def load_salary_data():
    """Load salary CSV and normalize team names to API abbreviations."""
    df = pd.read_csv(SALARY_CSV)
    df['Team'] = df['Team'].map(FULL_NAME_TO_ABBREV)
    df['Position'] = df['Position'].map(lambda p: 'G' if p == 'G' else ('D' if p == 'D' else 'F'))
    df['player_key'] = df['Player'].str.strip().str.upper()
    print(f"Loaded {len(df)} players from salary CSV")
    return df

In [8]:
# Cell 9 — Projection Engine

def calculate_projections(stats_df, games_count, multipliers):
    """Calculate projected fantasy points for each player.
    
    Formula: (goals_per_game * 2 + assists_per_game * 1) * games_this_week * multiplier
    """
    df = stats_df.copy()
    df['games_this_week'] = df['team'].map(games_count).fillna(0).astype(int)
    df['multiplier'] = df['team'].map(multipliers).fillna(1.0)
    df['proj_fantasy_pts'] = (
        (df['goals_per_game'] * 2 + df['assists_per_game'] * 1)
        * df['games_this_week']
        * df['multiplier']
    )
    df['player_key'] = df['player_name'].str.strip().str.upper()
    return df

In [9]:
# Cell 10 — Goalie Estimation

def estimate_team_goaltending_points(multipliers, games_count, standings_df,
                                      win_points=2, ot_loss_points=1,
                                      shutout_bonus=2, avg_ot_loss_freq=0.1,
                                      avg_shutout_freq=0.05):
    """Estimate goaltending fantasy points per team for the week.
    
    Win probability per game = team's own point_pctg * schedule multiplier (capped at 0.9).
    This means good teams get more projected goalie wins, and weak opponents boost it further.
    """
    pctg_lookup = dict(zip(standings_df['team'], standings_df['point_pctg']))
    goaltending_data = {}

    for team, multiplier in multipliers.items():
        games = games_count.get(team, 0)
        own_pctg = pctg_lookup.get(team, 0.5)
        win_prob = min(own_pctg * multiplier, 0.9)

        projected_wins = games * win_prob
        projected_ot_losses = games * avg_ot_loss_freq
        projected_shutouts = games * avg_shutout_freq

        total_points = (
            projected_wins * win_points
            + projected_ot_losses * ot_loss_points
            + projected_shutouts * shutout_bonus
        )
        goaltending_data[team] = (total_points, games)

    return goaltending_data

In [10]:
# Cell 11 — Data Assembly Pipeline

def _last_name_key(name):
    """Extract uppercase last name for fallback matching."""
    parts = name.strip().split()
    return parts[-1].upper() if parts else ''


def build_optimizer_input(start_date=None):
    """Orchestrate all data loading and produce a DataFrame ready for the optimizer."""
    # 1. Player stats
    stats_df = fetch_all_player_stats()

    # 2. Standings
    standings_df = fetch_standings()

    # 3. Schedule (always live)
    games_count, opponents = fetch_weekly_schedule(start_date)

    # 4. Multipliers
    multipliers = calculate_multipliers(standings_df, opponents)

    # 5. Projections
    proj_df = calculate_projections(stats_df, games_count, multipliers)

    # 6. Salary data
    salary_df = load_salary_data()

    # 7. Merge stats with salary (match on uppercase name + team)
    proj_cols = ['player_key', 'team', 'games_this_week', 'multiplier',
                 'proj_fantasy_pts', 'goals_per_game', 'assists_per_game']
    merged = salary_df.merge(
        proj_df[proj_cols],
        left_on=['player_key', 'Team'],
        right_on=['player_key', 'team'],
        how='left'
    )

    # Fallback: for unmatched players, try matching on last name + team
    # (handles Mitchell/Mitch Marner, John-Jason/JJ Peterka, etc.)
    unmatched_idx = merged[merged['proj_fantasy_pts'].isna()].index
    if len(unmatched_idx) > 0:
        merged['last_name_key'] = merged['Player'].apply(_last_name_key)
        proj_df['last_name_key'] = proj_df['player_name'].apply(_last_name_key)

        # Only attempt fallback where last name + team is unique in BOTH datasets
        api_counts = proj_df.groupby(['last_name_key', 'team']).size()
        unique_api = set(api_counts[api_counts == 1].index)

        salary_counts = merged.groupby(['last_name_key', 'Team']).size()
        unique_salary = set(salary_counts[salary_counts == 1].index)

        # Build lookup from unique API players
        fb_lookup = {}
        for _, row in proj_df.iterrows():
            key = (row['last_name_key'], row['team'])
            if key in unique_api:
                fb_lookup[key] = row

        fb_count = 0
        fb_names = []
        for idx in unmatched_idx:
            row = merged.loc[idx]
            key = (row['last_name_key'], row['Team'])
            if key in unique_api and key in unique_salary and key in fb_lookup:
                api_row = fb_lookup[key]
                merged.loc[idx, 'games_this_week'] = api_row['games_this_week']
                merged.loc[idx, 'multiplier'] = api_row['multiplier']
                merged.loc[idx, 'proj_fantasy_pts'] = api_row['proj_fantasy_pts']
                merged.loc[idx, 'goals_per_game'] = api_row['goals_per_game']
                merged.loc[idx, 'assists_per_game'] = api_row['assists_per_game']
                fb_count += 1
                fb_names.append(row['Player'])

        if fb_count > 0:
            print(f"Fallback matched {fb_count} players by last name: {', '.join(fb_names)}")

        merged.drop(columns=['last_name_key'], inplace=True)

    # Fill unmatched (no stats) with 0 projections
    merged['proj_fantasy_pts'] = merged['proj_fantasy_pts'].fillna(0)
    merged['games_this_week'] = merged['games_this_week'].fillna(0).astype(int)

    # 8. Injuries
    injuries_df = get_current_injuries()
    if not injuries_df.empty:
        injured_keys = set(
            injuries_df['Player'].str.strip().str.upper() + '|' + injuries_df['Team']
        )
        merged['Injured'] = (
            merged['player_key'] + '|' + merged['Team']
        ).isin(injured_keys)
        injured_count = merged['Injured'].sum()
        merged.loc[merged['Injured'], 'proj_fantasy_pts'] = 0
        print(f"Zeroed projections for {injured_count} injured players")
    else:
        merged['Injured'] = False

    # 9. Goalie rows
    goalie_data = estimate_team_goaltending_points(multipliers, games_count, standings_df)
    goalie_rows = []
    for team, (pts, games) in goalie_data.items():
        goalie_rows.append({
            'Player': f"{team} Goalie",
            'Team': team,
            'Position': 'G',
            'pv': 0,
            'proj_fantasy_pts': pts,
            'games_this_week': games,
            'Injured': False,
            'player_key': f"{team} GOALIE",
        })
    goalie_df = pd.DataFrame(goalie_rows)

    # Combine skaters + goalies
    result = pd.concat([merged, goalie_df], ignore_index=True)

    # Filter to only players on teams playing this week
    result = result[result['games_this_week'] > 0].copy()

    skater_count = len(result[result['Position'] != 'G'])
    goalie_count = len(result[result['Position'] == 'G'])
    print(f"\nOptimizer input ready: {skater_count} skaters + {goalie_count} goalies")
    return result

In [11]:
# Cell 12 — PuLP Optimizer

def select_best_team(df, max_cost=MAX_COST, min_cost=MIN_COST,
                     num_forwards=NUM_FORWARDS, num_defensemen=NUM_DEFENSEMEN,
                     num_goalies=NUM_GOALIES, max_players_per_team=MAX_PLAYERS_PER_TEAM):
    """Use PuLP to find the optimal lineup given constraints."""
    prob = pulp.LpProblem("FantasyHockeyTeam", pulp.LpMaximize)
    player_vars = pulp.LpVariable.dicts("player", df.index, cat="Binary")

    # Objective: maximize projected fantasy points
    prob += pulp.lpSum(df['proj_fantasy_pts'][i] * player_vars[i] for i in df.index)

    # Salary constraints
    prob += pulp.lpSum(df['pv'][i] * player_vars[i] for i in df.index) <= max_cost
    prob += pulp.lpSum(df['pv'][i] * player_vars[i] for i in df.index) >= min_cost

    # Position constraints
    prob += pulp.lpSum(player_vars[i] for i in df[df['Position'] == 'F'].index) == num_forwards
    prob += pulp.lpSum(player_vars[i] for i in df[df['Position'] == 'D'].index) == num_defensemen
    prob += pulp.lpSum(player_vars[i] for i in df[df['Position'] == 'G'].index) == num_goalies

    # Max players per team
    for team in df['Team'].unique():
        team_idx = df[df['Team'] == team].index
        prob += pulp.lpSum(player_vars[i] for i in team_idx) <= max_players_per_team

    # Max 1 defenseman per team
    for team in df['Team'].unique():
        d_idx = df[(df['Team'] == team) & (df['Position'] == 'D')].index
        prob += pulp.lpSum(player_vars[i] for i in d_idx) <= 1

    # Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    selected = [i for i in df.index if player_vars[i].varValue == 1]
    return df.loc[selected]

In [12]:
# Cell 13 — Display Helpers

def format_lineup(lineup_df):
    """Format and display the optimal lineup."""
    if lineup_df is None or lineup_df.empty:
        print("No lineup found!")
        return pd.DataFrame()

    display_columns = ['Team', 'Injured', 'Player', 'Position',
                       'games_this_week', 'proj_fantasy_pts', 'pv']
    # Only show columns that exist
    display_columns = [c for c in display_columns if c in lineup_df.columns]

    position_order = {'F': 2, 'D': 1, 'G': 0}
    sorted_lineup = (
        lineup_df[display_columns]
        .assign(pos_order=lineup_df['Position'].map(position_order))
        .sort_values(['pos_order', 'proj_fantasy_pts'], ascending=[True, False])
        .drop('pos_order', axis=1)
        .round(2)
    )

    # Summary
    total_pts = lineup_df['proj_fantasy_pts'].sum()
    total_cost = lineup_df['pv'].sum()
    n_f = (lineup_df['Position'] == 'F').sum()
    n_d = (lineup_df['Position'] == 'D').sum()
    n_g = (lineup_df['Position'] == 'G').sum()

    print(f"\n{'='*60}")
    print(f"OPTIMAL LINEUP")
    print(f"{'='*60}")
    print(f"Total Projected Points: {total_pts:.2f}")
    print(f"Total Salary: ${total_cost:.2f}M  (Cap: ${MAX_COST}M, Floor: ${MIN_COST:.2f}M)")
    print(f"Roster: {n_f}F / {n_d}D / {n_g}G")
    print(f"{'='*60}\n")

    return sorted_lineup

## Run the Optimizer

The cell below will:
1. Fetch (or use cached) player stats from the NHL API
2. Get the current week's schedule
3. Scrape injury data from CBS Sports
4. Merge with salary data
5. Run the PuLP optimizer to find the best 12-player lineup

In [13]:
# Cell 15 — Main Execution

# Build optimizer input (pass a date string like '2026-03-09' for mid-week runs)
optimizer_df = build_optimizer_input()

# Run optimizer
optimal_lineup = select_best_team(optimizer_df)

# Display results
format_lineup(optimal_lineup)

Using cached player stats (735 players)
Using cached standings (32 teams)
Fetching schedule starting 2026-03-09...
Found 56 games this week, 32 teams playing
Loaded 730 players from salary CSV
Fallback matched 24 players by last name: Mitchell Marner, Joshua Norris, John-Jason Peterka, Matthew Beniers, Cameron York, Artyom Zub, Christopher Tanev, Jacob Middleton, William Borgen, Thomas Novak, Janis Jérôme Moser, Nicklaus Perbix, Yegor Chinakhov, Alexei Toropchenko, Yegor Zamula, Benjamin Kindel, Maxim Shabanov, Fyodor Svechkov, Josh Mahura, Joseph Veleno, Matthew Savoie, Zack Bolduc, Maxwell Crozier, Joshua Dunne
Found 98 injured players
Zeroed projections for 85 injured players

Optimizer input ready: 688 skaters + 32 goalies

OPTIMAL LINEUP
Total Projected Points: 70.18
Total Salary: $67.81M  (Cap: $70.5M, Floor: $63.45M)
Roster: 6F / 4D / 2G



,Team,Injured,Player,Position,games_this_week,proj_fantasy_pts,pv
756,MIN,False,MIN Goalie,G,4,5.93,0.00
746,MTL,False,MTL Goalie,G,4,5.39,0.00
20,CBJ,False,Zach Werenski,D,4,5.55,9.58
14,EDM,False,Evan Bouchard,D,4,4.86,10.50
83,MIN,False,Quinn Hughes,D,4,4.60,7.85
535,MTL,False,Lane Hutson,D,4,4.41,0.95
3,EDM,False,Connor McDavid,F,4,7.64,12.50
114,MIN,False,Matt Boldy,F,4,6.89,7.00
35,MIN,False,Kirill Kaprizov,F,4,6.76,9.00
517,SJS,False,Macklin Celebrini,F,4,6.46,0.98


In [14]:
# Cell 16 — Optional Debug / Explore

# Uncomment any of these to inspect data:

# Top 20 projected skaters
# optimizer_df[optimizer_df['Position'] != 'G'].nlargest(20, 'proj_fantasy_pts')[['Player', 'Team', 'Position', 'games_this_week', 'proj_fantasy_pts', 'pv']]

# All goalies
# optimizer_df[optimizer_df['Position'] == 'G'].nlargest(10, 'proj_fantasy_pts')[['Player', 'Team', 'proj_fantasy_pts']]

# Teams with most games this week
# optimizer_df.groupby('Team')['games_this_week'].first().sort_values(ascending=False)